# Exercise 5 — A Small Explicit Dynamics Code

**Companion to Lecture 5** (Mass, Damping, and Time) of the lecture notes *Essential Concepts in FEM
and Physics-Based ML*.

In this exercise you **write the code** yourself: an assembly routine for a chain of springs and
masses, an eigenvalue solver for natural frequencies and modes, and an explicit time stepper.
Then you use it on a bar: convergence of frequencies, the stability limit, a travelling wave,
and the question of how many modes a response really needs.

Only `numpy` and `matplotlib`. Fill in every `...`; each part ends with a ✅ check cell.

---
*Author: Ramses Sala · Copyright © 2025 Ramses Sala · Part of the course* Essential Concepts in FEM and Physics-Based ML. *Created with assistance of Claude Code for text, image and code refinement and generation.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=5, suppress=True)


---
## Part 1 — Assembly of a chain

A chain of $n$ springs: spring $e$ connects node $e$ and node $e+1$, node 0 is fixed. A point mass
sits on each free node. Write a function that returns the stiffness matrix of the **free** nodes
($n \times n$) and the **diagonal** mass matrix, stored simply as a vector `m` of length $n$.

Reuse the assembly rule of Lecture 1: add $k\left[\begin{smallmatrix}1&-1\\-1&1\end{smallmatrix}\right]$ into
rows and columns $(e, e+1)$ of the full matrix, then remove the row and column of the fixed node.

*Why a vector for the mass?* A diagonal matrix carries $n$ numbers, not $n^2$; and "multiply by
$\mathbf M^{-1}$" becomes an element-wise division `x / m`.

In [ ]:
def assemble_chain(k_list, m_list):
    '''k_list: stiffnesses of the n springs; m_list: masses on the n free nodes.
    Returns K (n x n, free nodes only) and m (vector of length n).'''
    n = len(k_list)
    K_full = np.zeros((n + 1, n + 1))
    for e, k in enumerate(k_list):
        ...                                   # TODO: add the element matrix
    K = ...                                   # TODO: remove the fixed node 0
    m = np.asarray(m_list, dtype=float)
    return K, m

K2, m2 = assemble_chain([1.0, 1.0], [1.0, 1.0])
K2, m2

In [ ]:
# ✅ check — Part 1
assert np.allclose(K2, [[2, -1], [-1, 1]]), "The two-spring chain of Lecture 1 should give [[2,-1],[-1,1]]."
K3, _ = assemble_chain([3.0, 2.0, 1.0], [1, 1, 1])
assert np.allclose(K3, [[5, -2, 0], [-2, 3, -1], [0, -1, 1]])
print("OK. Same assembly as in Lecture 1 — dynamics adds a mass vector, nothing else.")

---
## Part 2 — Natural frequencies and modes

The eigenproblem $(\mathbf K - \omega^2\mathbf M)\boldsymbol\phi = \mathbf 0$ has a *diagonal* $\mathbf M$. That allows a trick
that turns it into a standard symmetric eigenproblem: with $\mathbf M^{-1/2} = \operatorname{diag}(1/\sqrt{m_i})$,
$$\mathbf A = \mathbf M^{-1/2}\,\mathbf K\,\mathbf M^{-1/2}, \qquad \mathbf A\,\mathbf y = \omega^2\,\mathbf y, \qquad \boldsymbol\phi = \mathbf M^{-1/2}\,\mathbf y .$$
`np.linalg.eigh` solves symmetric problems and returns eigenvalues in ascending order.

1. Implement `natural_frequencies(K, m)` returning $\omega$ (ascending) and the modes as columns of `Phi`.
2. Test on the two-mass chain: the lecture gives $\omega = 0.618$ and $1.618$.
3. The modes from this recipe are **mass-normalized**: $\boldsymbol\Phi^T\mathbf M\boldsymbol\Phi = \mathbf I$. Check it.

In [ ]:
def natural_frequencies(K, m):
    s = 1/np.sqrt(m)
    A = ...                      # TODO: M^(-1/2) K M^(-1/2)   (hint: s[:, None] * K * s[None, :])
    lam, Y = np.linalg.eigh(A)
    omega = ...                  # TODO
    Phi = ...                    # TODO: back-transform the eigenvectors
    return omega, Phi

omega2, Phi2 = natural_frequencies(K2, m2)
omega2

In [ ]:
# ✅ check — Part 2
assert np.allclose(omega2, [0.618034, 1.618034], atol=1e-6), "Expect (3 -/+ sqrt5)/2 under the square root."
assert np.allclose(Phi2.T @ (m2[:, None]*Phi2), np.eye(2)), "Modes should satisfy Phi^T M Phi = I."
ratio = Phi2[0, :] / Phi2[1, :]
print("mode shapes (first entry / second entry):", ratio, "  lecture: 0.618 and -1.618")

---
## Part 3 — A bar as a chain: how good are the frequencies?

A bar fixed at $x=0$, free at $x=L$, cut into `ne` elements of length $h$: spring stiffness $EA/h$,
**lumped** masses $\rho A h$ on interior nodes and $\rho A h/2$ on the end node. Units with
$EA = \rho A = L = 1$, so the wave speed is $c = 1$ and the exact frequencies are
$\omega_i = (2i-1)\,\pi/2$.

1. Write `bar_chain(ne)` using `assemble_chain`.
2. Reproduce the lecture table ($\omega_1$ = 1.414, 1.531, 1.553 for 1, 2, 3 elements).
3. For `ne = 20`, plot the first three modes against $\sin\bigl((2i-1)\pi x/2\bigr)$ (scale each so the
   tip value matches), and print the relative error of the first five frequencies. Which modes can you trust?

In [ ]:
def bar_chain(ne, EA=1.0, rhoA=1.0, L=1.0):
    h = L/ne
    k_list = ...                 # TODO
    m_list = ...                 # TODO: lumped masses (interior nodes, then the end node)
    return assemble_chain(k_list, m_list)

w1 = [natural_frequencies(*bar_chain(ne))[0][0] for ne in (1, 2, 3)]
print("omega_1 for 1, 2, 3 elements:", np.round(w1, 3), "  exact:", round(np.pi/2, 3))

ne = 20
K20, m20 = bar_chain(ne)
om20, Phi20 = natural_frequencies(K20, m20)
x = np.linspace(1/ne, 1, ne)
exact = (2*np.arange(1, 6) - 1)*np.pi/2
print("relative error of the first five frequencies:", np.round((om20[:5] - exact)/exact, 4))
for i in range(3):
    plt.plot(x, Phi20[:, i]/Phi20[-1, i], 'o', ms=3)
    plt.plot(x, np.sin((2*i + 1)*np.pi*x/2)/np.sin((2*i + 1)*np.pi/2), 'k-', lw=0.8)
plt.xlabel('x'); plt.title('first three modes (dots) and exact shapes'); plt.show()

In [ ]:
# ✅ check — Part 3
assert np.allclose(w1, [1.41421, 1.53073, 1.55291], atol=1e-4), "Compare with the table in Section 5.8."
assert abs(om20[0] - np.pi/2)/(np.pi/2) < 1e-3
print("OK. Lowest modes: excellent. Highest modes of the mesh: poor. Trust the lower spectrum.")

---
## Part 4 — An explicit time stepper

Implement the central-difference method of Lecture 5 for $\mathbf M\ddot{\mathbf u} + \mathbf C\dot{\mathbf u} + \mathbf K\mathbf u = \mathbf f(t)$,
with a **diagonal** damping $\mathbf C$ stored as a vector `c` (e.g. mass-proportional, $c_i = \alpha m_i$):
$$\Bigl(\frac{m}{\Delta t^2} + \frac{c}{2\Delta t}\Bigr)\,u^{n+1} = f^n - \mathbf K u^n + \frac{2m}{\Delta t^2}u^n - \Bigl(\frac{m}{\Delta t^2} - \frac{c}{2\Delta t}\Bigr)u^{n-1}$$
(element-wise in $m$ and $c$ — no linear system!). Start with
$\mathbf u^{-1} = \mathbf u^0 - \Delta t\,\dot{\mathbf u}^0 + \tfrac12\Delta t^2\,\ddot{\mathbf u}^0$, where $\ddot{\mathbf u}^0 = (\mathbf f(0) - \mathbf K\mathbf u^0 - \mathbf c\,\dot{\mathbf u}^0)/\mathbf m$.

**Store every state.** Return the time vector and a matrix `U` whose column `n` is $\mathbf u^n$. Such a
matrix of computed states is called a **snapshot matrix**; we will come back to it.

Test: one mass, $k = m = 1$, released from $u^0 = 1$, $\Delta t = 0.5$ — the lecture's hand table
(1, 0.875, 0.5313, 0.0547, −0.4355).

In [ ]:
def central_difference(K, m, f_func, u0, v0, dt, nsteps, c=None):
    c = np.zeros_like(m) if c is None else c
    a0 = ...                                  # TODO: initial acceleration
    u_prev = ...                              # TODO: u^{-1}
    u = u0.astype(float).copy()
    U = np.zeros((len(m), nsteps + 1)); U[:, 0] = u
    for n in range(nsteps):
        t = n*dt
        rhs = ...                             # TODO: right-hand side (vector)
        u_new = ...                           # TODO: divide element-wise
        u_prev, u = u, u_new
        U[:, n + 1] = u
    return np.arange(nsteps + 1)*dt, U

t1, U1 = central_difference(np.array([[1.0]]), np.array([1.0]), lambda t: np.zeros(1),
                            np.array([1.0]), np.zeros(1), 0.5, 4)
U1

In [ ]:
# ✅ check — Part 4
assert np.allclose(U1[0], [1, 0.875, 0.53125, 0.0546875, -0.43554688], atol=1e-7), "Compare with Section 5.7."
print("OK: your stepper reproduces the hand table.")

---
## Part 5 — The stability limit, found by experiment

For the 10-element bar compute $\omega_{\max}$ and the limit $\Delta t_{\rm crit} = 2/\omega_{\max}$.
Release the bar from a linear initial displacement and run 400 steps with $\Delta t = 0.95\,\Delta t_{\rm crit}$
and with $\Delta t = 1.05\,\Delta t_{\rm crit}$. Print the largest $|u|$ of each run.

Compare $\Delta t_{\rm crit}$ with $h/c$. What does the result say about refining the mesh?

In [ ]:
K10, m10 = bar_chain(10)
omega10, _ = natural_frequencies(K10, m10)
dt_crit = ...                                 # TODO
u_init = np.linspace(0.1, 1.0, 10)
results = {}
for factor in (0.95, 1.05):
    _, U = ...                                # TODO: run 400 steps, no load, released from u_init
    results[factor] = np.abs(U).max()
print(f"dt_crit = {dt_crit:.4f}   h/c = {0.1:.4f}")
print("max |u|:", results)

In [ ]:
# ✅ check — Part 5
assert abs(dt_crit - 0.1003) < 1e-3
assert results[0.95] < 1.5 and results[1.05] > 1e3, "Below the limit: bounded. Above: explosive growth."
print("OK. Five percent too large a step, and the answer is off by more than 100 orders of magnitude.")

---
## Part 6 — A sudden load, and a wave you can watch

20 elements, force $F = 1$ switched on at the free end at $t = 0$ and held. Run to $t = 8$ with
$\Delta t = 0.9\,h$, storing all states.

1. Plot the tip displacement against time. Static value: $FL/EA = 1$. How large is the overshoot?
2. Plot the displacement **along the bar** at $t = 0.25, 0.5, 0.75, 1.0$. You will see the load
   travel into the bar as a wave front, at speed $c = 1$.
3. Find the first time at which the middle node (x = 0.5) moves by more than 0.01.
4. Repeat with damping $c_i = 0.5\,m_i$ up to $t = 40$: where does the tip end up?

In [ ]:
ne = 20; h = 1/ne
K20, m20 = bar_chain(ne)
F = np.zeros(ne); F[-1] = 1.0
dt = 0.9*h
t6, U6 = ...                                  # TODO: undamped run to t = 8, starting at rest
tip_max = U6[-1].max()
mid = U6[ne//2 - 1]
t_arrive = ...                                # TODO: first time with mid > 0.01

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(t6, U6[-1]); ax[0].axhline(1, ls='--', c='k'); ax[0].set_xlabel('t'); ax[0].set_title('tip')
x = np.linspace(h, 1, ne)
for tt in (0.25, 0.5, 0.75, 1.0):
    ax[1].plot(x, U6[:, int(round(tt/dt))], label=f't={tt}')
ax[1].legend(); ax[1].set_xlabel('x'); ax[1].set_title('displacement along the bar'); plt.show()

td, Ud = ...                                  # TODO: damped run to t = 40 with c = 0.5*m20
print(f"tip max = {tip_max:.3f}   wave reaches x=0.5 at t = {t_arrive:.2f}   damped final tip = {Ud[-1, -1]:.4f}")

In [ ]:
# ✅ check — Part 6
assert 1.9 < tip_max < 2.05, "A suddenly applied load gives about twice the static displacement."
assert 0.45 < t_arrive < 0.6, "The front travels at c = 1: it needs about 0.5 to reach the middle."
assert abs(Ud[-1, -1] - 1.0) < 0.01, "With damping the bar settles on the static solution."
print("OK. Nothing happens at x = 0.5 before the wave gets there — information has a speed.")

---
## Part 7 — How many modes does this response need?

The snapshot matrix `U6` from Part 6 has 20 rows (nodes) and about 180 columns (time instants).
Every column is a displacement field. Expand it in the mass-normalized modes of Part 3:
$$\mathbf U = \boldsymbol\Phi\,\mathbf Q, \qquad \mathbf Q = \boldsymbol\Phi^T\mathbf M\,\mathbf U \quad(\text{because } \boldsymbol\Phi^T\mathbf M\boldsymbol\Phi = \mathbf I).$$
Row $i$ of $\mathbf Q$ is the modal coordinate $q_i(t)$.

1. Compute `Q`.
2. Reconstruct the response with only the first $r$ modes, $\mathbf U_r = \boldsymbol\Phi_{:,1:r}\,\mathbf Q_{1:r,:}$, and the relative error
   $\|\mathbf U - \mathbf U_r\| / \|\mathbf U\|$ for $r = 1, 2, 3, 4, 6, 10, 20$.
3. Plot $q_1(t)$, $q_2(t)$, $q_3(t)$. Which modes carry the response?

In [ ]:
om, Phi = natural_frequencies(K20, m20)
Q = ...                                       # TODO: modal coordinates of all snapshots
errors = {}
for r in (1, 2, 3, 4, 6, 10, 20):
    U_r = ...                                 # TODO
    errors[r] = np.linalg.norm(U6 - U_r)/np.linalg.norm(U6)
print({r: round(e, 4) for r, e in errors.items()})
for i in range(3):
    plt.plot(t6, Q[i], label=f'q_{i+1}')
plt.legend(); plt.xlabel('t'); plt.title('modal coordinates'); plt.show()

In [ ]:
# ✅ check — Part 7
assert errors[1] < 0.2 and errors[3] < 0.05 and errors[20] < 1e-10
assert all(errors[a] >= errors[b] for a, b in [(1, 2), (2, 3), (3, 4), (4, 6), (6, 10)])
print("OK. One mode out of twenty already reproduces this response with a relative error of about 13 %; three modes, about 3 %.")

---
## Closing reflections

1. In Part 4 not a single linear system was solved. Where exactly did the diagonal mass matrix make
   that possible, and what would change with the consistent mass matrix?
2. Halve the element size of Part 6. By what factor does the computing time grow (count element
   evaluations, not seconds)?
3. Part 7 needed the modes — an eigenvalue problem — before it could compress the response. Suppose
   you only had the snapshot matrix `U6`, from a simulation or from measurements, and no model to
   compute modes from. Could you still find a handful of shapes that describe all columns well?
   *(This question is the starting point of Lecture D on model reduction; no need to
   answer it fully now.)*